# Partie III complémentaire — Mini système Seq2Seq

Ce notebook présente un mini système encodeur-décodeur récurrent pour illustrer :

- vocabulaire source/cible ;
- tokens spéciaux `<sos>`, `<eos>`, `<pad>`, `<unk>` ;
- encodeur GRU ;
- décodeur GRU ;
- teacher forcing ;
- décodage glouton ;
- beam search ;
- score BLEU-1 simplifié.

Le corpus intégré est volontairement petit afin que l'expérience soit exécutable sur CPU. Pour un vrai projet de traduction, il faut remplacer ce corpus par un corpus parallèle réel comme Tatoeba.

In [ ]:
from pathlib import Path
import os, sys, json

# Exécuter ce notebook depuis le dossier project ou depuis project/notebooks fonctionne.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Python path ready")

In [ ]:
from types import SimpleNamespace
import torch
import pandas as pd

from src.common import ensure_dir, get_device, set_seed, save_json
from part3_rnn.tiny_seq2seq import (
    prepare_data,
    Encoder,
    Decoder,
    Seq2Seq,
    train,
    greedy_decode,
    beam_search_decode,
    bleu_unigram,
    tokenize,
    PAIRS,
    PAD, SOS, EOS,
)

set_seed(42)
device = get_device()
print("Device utilisé:", device)

## 1. Paramètres et données

Le modèle est volontairement petit : il sert à démontrer la logique Seq2Seq, pas à battre un système de traduction moderne.

In [ ]:
args = SimpleNamespace(
    epochs=300,
    batch_size=16,
    max_len=10,
    embedding_dim=32,
    hidden_dim=64,
    teacher_forcing=0.5,
    lr=1e-3,
)

out_dir = ensure_dir(ROOT / "results" / "part3_seq2seq")
loader, src_vocab, tgt_vocab = prepare_data(args.max_len, args.batch_size)

print("Vocabulaire source:", len(src_vocab))
print("Vocabulaire cible:", len(tgt_vocab))
print("Nombre de paires:", len(loader.dataset))

## 2. Construction du modèle encodeur-décodeur

L'encodeur transforme la phrase source en état caché. Le décodeur utilise cet état pour générer progressivement la phrase cible.

In [ ]:
encoder = Encoder(len(src_vocab), args.embedding_dim, args.hidden_dim, src_vocab[PAD])
decoder = Decoder(len(tgt_vocab), args.embedding_dim, args.hidden_dim, tgt_vocab[PAD])
model = Seq2Seq(encoder, decoder, tgt_vocab[SOS], tgt_vocab[EOS], args.max_len).to(device)

print(model)

## 3. Entraînement avec teacher forcing

Le teacher forcing consiste à donner parfois au décodeur le vrai token précédent au lieu du token qu'il vient de prédire. Cela stabilise l'entraînement.

In [ ]:
history = train(model, loader, tgt_vocab[PAD], args, device)
torch.save(model.state_dict(), out_dir / "seq2seq_gru_state_dict.pth")
print("Modèle sauvegardé dans:", out_dir)

## 4. Greedy decoding vs beam search

Le décodage glouton choisit le meilleur mot à chaque étape. Le beam search conserve plusieurs hypothèses et sélectionne la séquence globale la plus prometteuse.

In [ ]:
examples = []
for src_sentence, tgt_sentence in PAIRS[:8]:
    greedy = greedy_decode(model, src_sentence, src_vocab, tgt_vocab, args.max_len, device)
    beam = beam_search_decode(model, src_sentence, src_vocab, tgt_vocab, args.max_len, device, beam_width=3)
    reference = tokenize(tgt_sentence)
    examples.append({
        "source": src_sentence,
        "reference": tgt_sentence,
        "greedy": " ".join(greedy),
        "beam_search": " ".join(beam),
        "bleu1_greedy": bleu_unigram(reference, greedy),
        "bleu1_beam": bleu_unigram(reference, beam),
    })

save_json({"history": history, "examples": examples}, out_dir / "seq2seq_results.json")
pd.DataFrame(examples)

## 5. Analyse critique à intégrer au rapport

Le Seq2Seq justifie le passage d'un simple classifieur récurrent vers une architecture encodeur-décodeur lorsque la sortie est elle-même une séquence. Cette architecture est pertinente pour la traduction, la génération ou la transformation de séquences. Toutefois, un petit modèle récurrent sans attention reste limité : il compresse toute la phrase source dans un état caché unique, ce qui devient insuffisant pour des phrases longues ou ambiguës.